# Notebook 6.4 — From one brain patch to a complete masked release
## Patch geometry, boundary coverage, and covariance-informed noise

**Research question:** How do patch shape and overlap change coverage, computational cost, and the noise field obtained by assembling independent covariance-informed patch draws?

This tutorial follows the single-patch imaging, eigenmode, and patch-sizing notebooks. It takes a configurable patch size as an **input**, not an optimum. The progression is: understand the spatial grid → cover the mask → reconstruct a known image → construct patch noise → measure what assembly changes.

We compare anisotropic patches with two physically cubic controls: equal requested volume and the shortest requested anisotropic side. Requested overlap is 0/25/50/75%. Rounded voxel sizes and strides are reported explicitly.

**Scope:** covariance and assembly characterization. This is not a DP-calibrated release mechanism. LOO changes describe this cohort, not global sensitivity; data-dependent masks and covariance require their own privacy treatment. Low measured attack advantage would not prove DP. Nothing here replaces or changes Baseline A.

**How to use:** put the supplied `privacy_harness.py` beside this notebook, install the packages below, and Run All. OASIS is the default. Use the explicit synthetic mode for a fast offline walkthrough. Each plot poses a question; read its interpretation before proceeding.

## 0 — Reproducible setup and adjustable research choices

If needed, run `%pip install numpy scipy pandas matplotlib nilearn nibabel` once in your notebook environment, then restart the kernel. The supplied harness is imported directly; `cov_diagnostics` is not required.

Change **PATCH_MM** while patch sizing remains work in progress. Cubes are defined in physical units, before voxel rounding. `DOWNSAMPLE` uses strided subsampling of every subject on the same grid: it retains whole-volume extent but changes spatial resolution, without antialiasing. It is a computational teaching setting, not a substitute for a native-resolution study. Set it to 1 for the latter and check memory/cost first. No interpolation or registration is performed.

In [ ]:
from pathlib import Path
import importlib.util, hashlib, json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, binary_erosion
from IPython.display import display

DATA_MODE = "oasis"                 # "oasis", "nifti", or explicitly "synthetic"
HARNESS_PATH = Path("privacy_harness.py")
DATA_DIR = None                     # shared Nilearn cache, or a path
N_FETCH, N_SUBJECTS = 60, 40
NIFTI_DIR = Path("aligned_subject_maps")
MASK_PATH = None                    # optional aligned NIfTI mask in real-data modes
DOWNSAMPLE = 4                      # OASIS/native-grid factor; synthetic stays at its own resolution
PATCH_MM = (12., 18., 24.)          # image-axis lengths; not necessarily anatomical x/y/z
OVERLAPS = (0., .25, .50, .75)
DRAWS = 80                         # increase for stable Monte Carlo estimates
RIDGE_FRACTION = .02                # positive isotropic component, no eigenmode truncation
TARGET_SD = .05                    # illustrative units of the output image; not epsilon
SEED = 2026
MAX_PATCHES = 100000                # explicit guard; never silently drop patches
OUT_DIR = Path("patch_results")
OUT_DIR.mkdir(exist_ok=True)

if not HARNESS_PATH.exists():
    matches = sorted(Path('.').glob('privacy_harness*.py'))
    if len(matches) == 1: HARNESS_PATH = matches[0]
    else: raise FileNotFoundError('Set HARNESS_PATH to the supplied privacy harness.')
spec = importlib.util.spec_from_file_location('ph', HARNESS_PATH)
ph = importlib.util.module_from_spec(spec); spec.loader.exec_module(ph)
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True, 'grid.alpha':.2})
COLORS = {'anisotropic':'#2878B5', 'equal_volume_cube':'#E07B39', 'minimum_side_cube':'#579D62'}
print('Harness:', HARNESS_PATH, '| SHA256:', hashlib.sha256(HARNESS_PATH.read_bytes()).hexdigest()[:16])
assert len(PATCH_MM)==3 and np.all(np.asarray(PATCH_MM)>0)
assert DRAWS >= 3 and N_SUBJECTS >= 3 and 0 < RIDGE_FRACTION <= 1 and TARGET_SD > 0
assert all(0 <= o < 1 for o in OVERLAPS)

## 1 — Load aligned subjects using the same harness contract

The harness contract is `(subjects, axis0, axis1, axis2)` plus a Boolean mask. We use `ph.load_oasis` / `ph.load_nifti_dir`, not `ph.oasis_source`, because the latter chooses one interior block. NIfTI headers supply spacing and verify matching affine matrices. This verifies the grid, **not anatomical registration**.

OASIS here means Nilearn's preprocessed gray-matter maps, not raw T1 images or fMRI. The default harness mask is finite voxels with a nonzero value in at least one subject. You may supply a more appropriate predefined mask; this changes the experimental domain and must be reported. No intensity standardization is added.

OASIS fetcher and dataset conditions: [Nilearn documentation](https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_oasis_vbm.html). The first fetch requires internet and may download more than the requested subject subset. This tutorial never switches datasets silently.

In [ ]:
def synthetic_volumes(n, seed):
    rng = np.random.default_rng(seed)
    shape = (24, 22, 20)
    q = np.indices(shape).astype(float)
    r = sum(((q[a]-(shape[a]-1)/2)/(shape[a]*.44))**2 for a in range(3))
    mask = r < 1
    mask[10:13, 9:12, 8:11] = False  # hole to make boundary handling visible
    base = .5*np.exp(-r/2)
    vols = np.stack([base + .12*gaussian_filter(rng.normal(size=shape), (1.,1.8,2.5))
                     + .03*rng.normal()*base for _ in range(n)])
    return vols.astype(np.float32), mask, np.array([3.,3.,3.]), np.diag([3.,3.,3.,1.])

if DATA_MODE == 'synthetic':
    volumes, mask, spacing, affine = synthetic_volumes(N_SUBJECTS, SEED)
    subject_files = []; factor = 1
else:
    import nibabel as nib
    if DATA_MODE == 'oasis':
        from nilearn import datasets
        oasis = datasets.fetch_oasis_vbm(n_subjects=N_FETCH, data_dir=DATA_DIR)
        subject_files = list(oasis.gray_matter_maps)
        volumes, mask = ph.load_oasis(n=N_FETCH, data_dir=DATA_DIR)
    elif DATA_MODE == 'nifti':
        subject_files = sorted(str(p) for p in NIFTI_DIR.glob('*.nii*'))
        volumes, mask = ph.load_nifti_dir(str(NIFTI_DIR))
    else: raise ValueError('Unknown DATA_MODE')
    first = nib.load(subject_files[0]); affine = first.affine.copy()
    for f in subject_files:
        img = nib.load(f)
        if img.shape != first.shape or not np.allclose(img.affine, affine, atol=1e-5):
            raise ValueError(f'Grid mismatch: {f}; align explicitly before this notebook.')
    if MASK_PATH is not None:
        mi = nib.load(str(MASK_PATH))
        if mi.shape != first.shape or not np.allclose(mi.affine, affine, atol=1e-5):
            raise ValueError('Mask grid mismatch')
        mv = mi.get_fdata(); mask = np.isfinite(mv) & (mv > 0)
    if not isinstance(DOWNSAMPLE, int) or DOWNSAMPLE < 1: raise ValueError('DOWNSAMPLE must be a positive integer')
    factor = DOWNSAMPLE
    volumes = volumes[:, ::factor, ::factor, ::factor].astype(np.float32)
    mask = mask[::factor, ::factor, ::factor]
    affine = affine @ np.diag([factor, factor, factor, 1.])
    spacing = np.linalg.norm(affine[:3,:3], axis=0)
    subject_files = subject_files[:N_SUBJECTS]

if len(volumes) < N_SUBJECTS: raise ValueError('Not enough subjects; lower N_SUBJECTS explicitly.')
volumes = volumes[:N_SUBJECTS]
mask = mask.astype(bool)
if not mask.any() or not np.isfinite(volumes[:, mask]).all(): raise ValueError('Empty mask or nonfinite masked data')
# Axis spacing assumes an orthogonal voxel grid; reject sheared grids for physical cube comparisons.
unit_axes = affine[:3,:3] / spacing
if not np.allclose(unit_axes.T @ unit_axes, np.eye(3), atol=1e-4): raise ValueError('Sheared affine: resample to an orthogonal grid first')
y = ph.pipeline_mean(volumes)
# Exact mean shortcut: matches ph.loo_artifacts without N repeated whole-volume deletions.
D = (y[None] - volumes)/(N_SUBJECTS-1)
valid_global = np.flatnonzero(mask)
Dflat = D.reshape(N_SUBJECTS, -1)[:, valid_global].astype(float)
yflat = y.ravel()[valid_global].astype(float)
Sflat = np.abs(Dflat).max(0)
lookup = np.full(mask.size, -1, dtype=int); lookup[valid_global] = np.arange(mask.sum())
print(f'Dataset={DATA_MODE}; subjects={N_SUBJECTS}; grid={mask.shape}; spacing={spacing} mm')
print(f'Masked voxels={mask.sum():,}; downsampling={factor}; loaded array={volumes.nbytes/2**20:.1f} MiB')
small = volumes[:, mask][:, :7]
_, direct_D, _ = ph.loo_artifacts(small, ph.pipeline_mean)
np.testing.assert_allclose(Dflat[:, :7], direct_D, atol=2e-7, rtol=1e-4)
print('Mean LOO shortcut agrees with harness on a checked subset.')

### What is covariance across here?

For this notebook, each row of `Dflat` is one subject's remove-one change to the **population mean**; each column is a masked voxel. Within a patch, covariance is estimated across those subject rows. It is not temporal fMRI covariance and not autocorrelation estimated from a single output image.

The signed changes follow the harness convention `mean(without i) − mean(all)`. For the mean, `D_i=(mean−X_i)/(N−1)`, hence the empirical covariance satisfies `Cov(D)=Cov(X)/(N−1)^2`. For a nonlinear pipeline, use `ph.loo_artifacts` and do not reuse this shortcut.

In [ ]:
z = int(np.argmax(mask.sum(axis=(0,1))))
fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, image, title in zip(axes, [np.where(mask,y,np.nan), mask], ['Mean gray-matter map', 'Fixed experimental mask']):
    im=ax.imshow(image[:,:,z].T, origin='lower', cmap='gray'); ax.set_title(title)
    ax.set_xlabel('image axis 0 (voxel)'); ax.set_ylabel('image axis 1 (voxel)'); ax.grid(False)
fig.suptitle(f'{DATA_MODE.upper()}: identical grid and mask for every comparison (slice {z})')
plt.tight_layout(); plt.show()

## 2 — Build the three geometries and report realized overlap

For requested physical lengths `(a,b,c)`, the equal-volume cube has side `(abc)^(1/3)`; the minimum-side cube has side `min(a,b,c)`. We round each physical length to its nearest positive voxel count. On an anisotropic grid a physically cubic request may therefore have unequal voxel counts.

Stride is `max(1, round(size*(1-overlap)))` on each axis. A single global lattice starts at image index zero; the last patch is clipped at the image boundary rather than shifted backwards. Consequently requested zero overlap really is zero overlap on this lattice. Patches intersecting the mask are kept, however little valid tissue they contain. Only valid mask voxels participate in covariance and assembly; outside-mask zeros are never treated as observations.

This favors guaranteed coverage over a fully-in-mask-only policy. Patch occupancy and small boundary patches are reported instead of silently excluded.

In [ ]:
requested = np.asarray(PATCH_MM)
physical = {'anisotropic': requested,
            'equal_volume_cube': np.repeat(np.prod(requested)**(1/3),3),
            'minimum_side_cube': np.repeat(requested.min(),3)}
geometries = {k: tuple(np.maximum(1,np.floor(v/spacing+.5).astype(int))) for k,v in physical.items()}
geometry_table = pd.DataFrame([{'shape':k, 'requested_mm':tuple(np.round(v,3)),
    'voxels':geometries[k], 'realized_mm':tuple(np.round(np.asarray(geometries[k])*spacing,3)),
    'requested_mm3':np.prod(v), 'realized_mm3':np.prod(np.asarray(geometries[k])*spacing)} for k,v in physical.items()])
display(geometry_table)

def build_tiles(shape, overlap):
    shape = np.asarray(shape); stride = np.maximum(1,np.floor(shape*(1-overlap)+.5).astype(int))
    tiles=[]; counts=np.zeros(mask.sum(),int)
    starts=[range(0,s,int(t)) for s,t in zip(mask.shape,stride)]
    for start in __import__('itertools').product(*starts):
        sl=tuple(slice(a,min(a+int(b),s)) for a,b,s in zip(start,shape,mask.shape))
        local_mask=mask[sl]
        if not local_mask.any(): continue
        coords=np.argwhere(local_mask)
        world=coords+np.asarray(start)
        g=lookup[np.ravel_multi_index(world.T, mask.shape)]
        actual_shape=local_mask.shape
        tiles.append({'g':g,'coords':coords,'extent':actual_shape, 'start':start,
                      'occupancy':len(g)/local_mask.size})
        counts[g]+=1
        if len(tiles)>MAX_PATCHES: raise RuntimeError('MAX_PATCHES exceeded; revise resolution/size before running')
    if np.any(counts==0): raise AssertionError('Coverage failure')
    return tiles, counts, stride

coverage_rows=[]; examples={}
for name, shape in geometries.items():
    for o in OVERLAPS:
        tiles, counts, stride=build_tiles(shape,o)
        coverage_rows.append({'shape':name,'overlap':o,'patches':len(tiles),'coverage':np.mean(counts>0),
            'mean_multiplicity':counts.mean(),'max_multiplicity':counts.max(),
            'median_occupancy':np.median([t['occupancy'] for t in tiles]),
            'small_patches_p_lt_N':np.mean([len(t['g'])<N_SUBJECTS for t in tiles]),
            'realized_overlap_axes':tuple(np.round(1-stride/np.asarray(shape),3)),
            'sum_patch_voxels':sum(len(t['g']) for t in tiles)})
        if name=='anisotropic': examples[o]=counts
coverage=pd.DataFrame(coverage_rows); display(coverage)
fig, axes=plt.subplots(1,2,figsize=(11,4))
for name in geometries:
    t=coverage[coverage['shape']==name]
    axes[0].plot(t.overlap*100,t.patches,'o-',label=name,color=COLORS[name])
    axes[1].plot(t.overlap*100,t.mean_multiplicity,'o-',label=name,color=COLORS[name])
axes[0].set_ylabel('Number of nonempty patches'); axes[1].set_ylabel('Mean patch contributions per voxel')
for ax in axes: ax.set_xlabel('Requested overlap (%)'); ax.legend(fontsize=8)
fig.suptitle('Coverage is complete; overlap changes the amount of repeated work')
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,len(OVERLAPS),figsize=(12,3.5),squeeze=False)
maxc=max(c.max() for c in examples.values())
for ax,(o,c) in zip(axes[0],examples.items()):
    image=np.full(mask.size,np.nan); image[valid_global]=c
    im=ax.imshow(image.reshape(mask.shape)[:,:,z].T,origin='lower',vmin=0,vmax=maxc,cmap='viridis')
    ax.set_title(f'{o:.0%} requested overlap'); ax.grid(False); ax.set_xlabel('axis 0'); ax.set_ylabel('axis 1')
fig.colorbar(im,ax=axes.ravel().tolist(),label='Patch contributions',shrink=.8)
plt.show()

**Read this result:** Coverage must be exactly 100% in every row. Multiplicity maps explain where subsequent averaging can alter noise. Overlap rounding can make two requested settings identical for small patches; interpret the realized values, not just the labels. If the shape controls round to the same voxel shape, this resolution cannot distinguish them.

## 3 — Verify assembly before generating noise

Uniform weights give each contributing patch equal importance. A positive separable sine taper gives more weight to its interior. Unlike a zero-ended Hann window, it does not lose singly covered edge voxels. At a clipped volume edge the taper is defined on the actual retained extent.

Weights are normalized **per voxel**. This is a partition of unity: reconstructing the same clean image through all patches should recover it to numerical precision. It proves correct assembly, not covariance preservation.

In [ ]:
def tile_weights(tile, blending):
    if blending=='uniform': return np.ones(len(tile['g']))
    if blending!='tapered': raise ValueError(blending)
    q=tile['coords']; ext=np.asarray(tile['extent'])
    return np.prod(np.sin(np.pi*(q+.5)/ext),axis=1)

def normalized_weights(tiles, blending):
    weights=[tile_weights(t,blending) for t in tiles]
    den=np.zeros(mask.sum())
    for t,w in zip(tiles,weights): den[t['g']]+=w
    assert np.all(den>0)
    return [w/den[t['g']] for t,w in zip(tiles,weights)]

reconstruction=[]
for name,shape in geometries.items():
    for o in OVERLAPS:
        tiles,_,_=build_tiles(shape,o)
        for blending in ['uniform','tapered']:
            a=normalized_weights(tiles,blending); rec=np.zeros_like(yflat)
            for t,w in zip(tiles,a): rec[t['g']]+=w*yflat[t['g']]
            err=np.max(np.abs(rec-yflat))
            np.testing.assert_allclose(rec,yflat,atol=1e-10)
            reconstruction.append((name,o,blending,err))
print('All clean-image reconstruction checks pass; worst absolute error:',max(r[3] for r in reconstruction))

## 4 — Construct an explicit covariance-informed Gaussian experiment

For patch `b`, center its LOO matrix `D_b` across subjects. Its empirical covariance is `C_b=D_bᵀD_b/(N−1)`. Add a fixed **relative ridge** using `m_b=trace(C_b)/p_b`:

`C̃_b = (1−λ) C_b + λ m_b I`.

Normalize its trace to `p_b × TARGET_SD²`. If the empirical trace is numerically zero, use an explicitly counted isotropic patch. This matches average marginal power, not each individual voxel's variance. It also makes tiny boundary patches well-defined.

We sample this covariance exactly using a subject-space factor and an independent diagonal term, without truncating eigenmodes or forming a dense whole-brain covariance. The positive ridge is retained. This computational factorization must not be confused with top-k eigenmode noise from notebook 6.1.

Independent patch draws are assembled as `z(v)=Σ_b a_b(v) z_b(v)`. Conditional on the fitted covariances, the assembled covariance is

`Cov(z(u),z(v)) = Σ_{b containing u,v} a_b(u)a_b(v) C̃_b(u,v)`.

Thus shared patches create cross-voxel covariance, while independently generated disjoint patches do not. The marginal variance is `Σ_b a_b(v)² C̃_b(v,v)`: averaging usually reduces it. We report this **uncorrected**, and optionally multiply each voxel by `TARGET_SD / sqrt(Var(z(v)))` to restore the target diagonal. This correction changes the assembled covariance; it does not restore every desired correlation or establish privacy.

In [ ]:
def patch_factor(g):
    M=Dflat[:,g]; M=M-M.mean(0)
    F=M/np.sqrt(N_SUBJECTS-1)
    m=np.sum(F*F)/len(g)
    if m <= np.finfo(float).tiny:
        return np.zeros_like(F), TARGET_SD, np.full(len(g),TARGET_SD**2), True
    factor=TARGET_SD/np.sqrt(m)
    F=F*np.sqrt(1-RIDGE_FRACTION)*factor
    sd=TARGET_SD*np.sqrt(RIDGE_FRACTION)
    diagonal=np.sum(F*F,axis=0)+sd**2
    return F,sd,diagonal,False

# Adjacent pairs are defined on the mask, including irregular boundaries.
pairs=[]
for axis in range(3):
    sl1=[slice(None)]*3; sl2=[slice(None)]*3
    sl1[axis]=slice(None,-1); sl2[axis]=slice(1,None)
    good=mask[tuple(sl1)] & mask[tuple(sl2)]
    ids=lookup.reshape(mask.shape)
    pairs.append((ids[tuple(sl1)][good],ids[tuple(sl2)][good]))
boundary = (mask & ~binary_erosion(mask)).ravel()[valid_global]

def pair_correlation(samples):
    M=samples-samples.mean(0)
    std=np.sqrt(np.mean(M*M,axis=0))
    out=[]
    for u,v in pairs:
        den=std[u]*std[v]; ok=den>1e-15
        out.append(np.mean(np.mean(M[:,u[ok]]*M[:,v[ok]],axis=0)/den[ok]) if ok.any() else np.nan)
    return np.asarray(out)

reference_corr=pair_correlation(Dflat)
noise_rows=[]; fit_rows=[]; example_maps={}
for ni,(name,shape) in enumerate(geometries.items()):
    for oi,o in enumerate(OVERLAPS):
        print('Noise experiment:',name, f'{o:.0%}',flush=True)
        tiles,counts,_=build_tiles(shape,o)
        for bi,blending in enumerate(['uniform','tapered']):
            t0=time.perf_counter(); weights=normalized_weights(tiles,blending)
            Z=np.zeros((DRAWS,mask.sum())); variance=np.zeros(mask.sum()); zero=0
            # Same local random stream for both weightings: differences isolate assembly weights.
            rng=np.random.default_rng(np.random.SeedSequence([SEED,ni,oi]))
            for t,a in zip(tiles,weights):
                g=t['g']; F,sd,diag,iszero=patch_factor(g); zero+=iszero
                local=rng.standard_normal((DRAWS,N_SUBJECTS))@F + sd*rng.standard_normal((DRAWS,len(g)))
                Z[:,g]+=local*a
                variance[g]+=a*a*diag
            assert np.all(variance>0)
            elapsed=time.perf_counter()-t0
            fit_rows.append({'shape':name,'overlap':o,'blending':blending,'seconds':elapsed,'isotropic_zero_trace_patches':zero})
            for corrected in [False,True]:
                W=Z*(TARGET_SD/np.sqrt(variance)) if corrected else Z
                predicted=np.full_like(variance,TARGET_SD**2) if corrected else variance
                empirical=W.var(0,ddof=1)
                corr=pair_correlation(W)
                noise_rows.append({'shape':name,'overlap':o,'blending':blending,'corrected':corrected,
                    'expected_total_power':predicted.sum(),'empirical_total_power':empirical.sum(),
                    'expected_variance_ratio':predicted.mean()/TARGET_SD**2,
                    'boundary_variance_ratio':predicted[boundary].mean()/TARGET_SD**2,
                    'interior_variance_ratio':predicted[~boundary].mean()/TARGET_SD**2 if (~boundary).any() else np.nan,
                    'corr_axis0':corr[0],'corr_axis1':corr[1],'corr_axis2':corr[2],
                    'mean_abs_corr_gap':np.nanmean(np.abs(corr-reference_corr)),
                    'relative_MSE':np.mean(W*W)/np.mean(yflat*yflat),
                    'variance_relative_L1_error':np.mean(np.abs(empirical-predicted))/predicted.mean()})
                if name=='anisotropic' and o in [min(OVERLAPS),max(OVERLAPS)] and blending=='uniform':
                    example_maps[(o,corrected)]=(W[0].copy(), predicted.copy())
noise_results=pd.DataFrame(noise_rows); timings=pd.DataFrame(fit_rows)
display(noise_results)
print('LOO reference adjacent correlations:',reference_corr)

## 5 — Evidence panel: does overlap dilute the noise?

The left plot uses the **analytic** marginal variance, avoiding Monte Carlo fluctuation. The right plot shows measured total power divided by its analytic expectation. With independent Gaussian draws its average should approach 1 as DRAWS increases. Keep this sampling check separate from the overlap effect.

Boundary and interior means are in the results table. There is no reason to assume they are identical: occupancy, local covariance and the number of contributing patches can differ.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name in geometries:
    t=noise_results[(noise_results['shape']==name)&(noise_results.blending=='uniform')&(~noise_results.corrected)]
    axes[0].plot(t.overlap*100,t.expected_variance_ratio,'o-',label=name,color=COLORS[name])
    axes[1].plot(t.overlap*100,t.empirical_total_power/t.expected_total_power,'o-',label=name,color=COLORS[name])
for ax in axes: ax.axhline(1,color='black',ls=':',lw=1); ax.set_xlabel('Requested overlap (%)'); ax.legend(fontsize=8)
axes[0].set_ylabel('Assembled variance / target variance'); axes[0].set_title('Uncorrected averaging changes marginal power')
axes[1].set_ylabel('Measured power / analytic power'); axes[1].set_title(f'{DRAWS} draws: check the sampler against the formula')
plt.tight_layout(); plt.show()

fig,axes=plt.subplots(2,2,figsize=(9,7))
for ax,((o,corrected),(draw,var)) in zip(axes.ravel(),example_maps.items()):
    arr=np.full(mask.size,np.nan); arr[valid_global]=var/TARGET_SD**2
    im=ax.imshow(arr.reshape(mask.shape)[:,:,z].T,origin='lower',vmin=0,vmax=max(1,max(v[1].max()/TARGET_SD**2 for v in example_maps.values())),cmap='viridis')
    ax.set_title(f'{o:.0%} overlap; '+('diagonal restored' if corrected else 'uncorrected')); ax.grid(False)
fig.colorbar(im,ax=axes.ravel().tolist(),label='Analytic voxel variance / target variance',shrink=.8)
fig.suptitle('Where assembly alters noise power — one common color scale'); plt.show()

**Interpretation:** inspect the computed ratios before concluding how strong dilution is. The corrected variance maps must equal one by construction. That confirms a diagonal correction, not a recovered full covariance. For a future rejection mechanism, per-patch acceptance also need not survive blending: signed patch draws can cancel.

## 6 — At restored marginal variance, is spatial structure preserved?

We now compare corrected noise across patch shapes and weighting rules. Correlation is estimated **across noise draws** separately for each pair of adjacent masked voxels, then averaged by image axis. The reference is the analogous covariance structure of cohort LOO changes, not spatial correlation of the mean image.

An empirical diagonal restoration preserves the fitted Gaussian field's correlations in theory (positive fixed voxel scaling); it is the patch assembly itself that alters them. The axis panels reveal anisotropy that one pooled score hides. The absolute gap summarizes discrepancy, not privacy or optimality.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4),sharey=True)
for axis,ax in enumerate(axes):
    ax.axhline(reference_corr[axis],color='black',ls='--',label='LOO reference')
    for name in geometries:
        for blending,ls in [('uniform','-'),('tapered',':')]:
            t=noise_results[(noise_results['shape']==name)&(noise_results.blending==blending)&noise_results.corrected]
            ax.plot(t.overlap*100,t[f'corr_axis{axis}'],'o',ls=ls,color=COLORS[name],label=f'{name}, {blending}')
    ax.set_xlabel('Requested overlap (%)'); ax.set_title(f'Image axis {axis}'); ax.set_ylim(-.15,1.05)
axes[0].set_ylabel('Mean adjacent-voxel correlation across draws'); axes[-1].legend(fontsize=6,loc='lower right')
fig.suptitle('Patch assembly changes correlation even when marginal variance is restored')
plt.tight_layout(); plt.show()

fig,axes=plt.subplots(1,2,figsize=(11,4))
for name in geometries:
    for blending,ls in [('uniform','-'),('tapered',':')]:
        t=noise_results[(noise_results['shape']==name)&(noise_results.blending==blending)&noise_results.corrected]
        axes[0].plot(t.overlap*100,t.mean_abs_corr_gap,'o',ls=ls,color=COLORS[name],label=f'{name}, {blending}')
    tt=timings[(timings['shape']==name)&(timings.blending=='uniform')]
    axes[1].plot(tt.overlap*100,tt.seconds,'o-',color=COLORS[name],label=name)
axes[0].set_ylabel('Mean absolute axis correlation gap'); axes[0].legend(fontsize=6)
axes[1].set_ylabel('Factor + sampling + assembly time (s)'); axes[1].legend(fontsize=8)
for ax in axes: ax.set_xlabel('Requested overlap (%)')
axes[0].set_title('Spatial fidelity at the same marginal noise scale'); axes[1].set_title('Cost includes every nonempty patch')
plt.tight_layout(); plt.show()

## 7 — Bridge back to the original single-patch harness

Whole-mask assembly and single-patch membership games answer different questions. Below, one interior anisotropic block uses the exact original `ArrayBlockSource → evaluate` path with iid and covariance Gaussian mechanisms. The total noise power is matched. These are **local reference experiments**, not attacks against the assembled whole-brain mechanism, and they do not use our ridge/assembly rule.

This optional cell is off by default because it is slower and requires a fully in-mask block. A failure to find such a block is informative for a thin or fragmented mask: reduce the patch size explicitly rather than changing the dataset. Boundary tiling above has no such requirement.

The local bridge explicitly uses `target_mode="k_sigma"`, the harness adversarial stress target. It is not a claim about held-out real-subject membership.


In [ ]:
RUN_LOCAL_HARNESS = False
if RUN_LOCAL_HARNESS:
    src=ph.ArrayBlockSource(volumes,mask,block_shape=geometries['anisotropic'],N=N_SUBJECTS,seed=SEED)
    local=[]
    for mechanism,fn in [('iid',ph.mech_iid),('covariance_gaussian',ph.mech_cov_gaussian)]:
        for seed in [1,2,3]:
            result=ph.evaluate(src,ph.pipeline_mean,fn,N=N_SUBJECTS,
                              noise_power=src.p*TARGET_SD**2,n_trials=800,n_rel=60,seed=seed, target_mode="k_sigma")
            local.append(dict(mechanism=mechanism,seed=seed,**result))
    local_results=pd.DataFrame(local)
    display(local_results)
    local_results.to_csv(OUT_DIR/'local_harness_results.csv',index=False)
else:
    print('Local harness experiment disabled. No whole-brain membership claim is made.')

## 8 — A reviewable summary and the next research decision

Read the table below alongside the figures. All settings remain visible; the notebook does not select an optimum automatically. A smaller correlation gap is one utility objective and can compete with runtime, native resolution, patch occupancy, and later privacy calibration.

**What this establishes:** the implemented tiling covers its specified mask; normalized weights reconstruct a known output; independent Gaussian patch covariance predicts assembled marginal variance; patch shape and overlap can be compared at restored marginal power.

**What remains open:** scientifically motivated patch sizes, mask selection, local versus global covariance structure, variation across cohorts and grid offsets, rejection/fallback effects, calibrated subject-level DP, and attacks on the actual assembled release. The fixed grid here controls placement rather than demonstrating robustness to it. Finite DRAWS introduces sampling error; repeat with larger DRAWS and different SEED before interpreting small correlation gaps.

**Suggested follow-up:** repeat the same analysis at native resolution and with candidate physical patch sizes from the preceding sizing notebook. Keep the cohort, mask, ridge, and target scale fixed. Then implement the assembled mechanism's IN/OUT worlds in the privacy harness, recomputing every data-dependent element in each world.

In [ ]:
summary=noise_results[noise_results.corrected].merge(timings,on=['shape','overlap','blending']).merge(
    coverage[['shape','overlap','patches','mean_multiplicity','median_occupancy']],on=['shape','overlap'])
display(summary[['shape','overlap','blending','patches','mean_multiplicity','median_occupancy',
                 'mean_abs_corr_gap','relative_MSE','seconds','isotropic_zero_trace_patches']])
for name,table in [('geometry',geometry_table),('coverage',coverage),('noise',noise_results),('timings',timings),('summary',summary)]:
    table.to_csv(OUT_DIR/f'{name}.csv',index=False)
manifest={'data_mode':DATA_MODE,'N_subjects':N_SUBJECTS,'N_fetch':N_FETCH,'patch_mm':PATCH_MM,
    'spacing_mm':spacing.tolist(),'grid':list(mask.shape),'downsample':factor,'overlaps':OVERLAPS,
    'draws':DRAWS,'ridge_fraction':RIDGE_FRACTION,'target_sd':TARGET_SD,'seed':SEED,
    'mask_voxels':int(mask.sum()),'mask_sha256':hashlib.sha256(mask.tobytes()).hexdigest(),
    'harness_sha256':hashlib.sha256(HARNESS_PATH.read_bytes()).hexdigest(),
    'subject_files':subject_files,'mask_path':str(MASK_PATH),'python':platform.python_version(),
    'numpy':np.__version__,'pandas':pd.__version__,'affine':affine.tolist()}
(OUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Complete coverage:',bool((coverage.coverage==1).all()))
print('Max clean reconstruction error:',max(r[3] for r in reconstruction))
raw=noise_results[(~noise_results.corrected)&(noise_results.blending=='uniform')]
print('Uncorrected mean variance / target, observed range:',raw.expected_variance_ratio.min(),raw.expected_variance_ratio.max())
print('Saved tables and reproducibility manifest to',OUT_DIR)

### Exercises for a researcher joining the project

1. Predict how zero overlap affects covariance between neighboring voxels on opposite sides of a patch boundary. Check whether the axis correlation plots are consistent with that prediction.
2. Increase DRAWS. Which quantities change through sampling error, and which remain fixed analytically?
3. Change PATCH_MM while keeping the data grid fixed. Are the two cubic controls still different after rounding? What changes in occupancy and runtime?
4. Compare uniform and tapered weighting before and after diagonal restoration. Explain why reconstruction of the clean image can be exact while covariance reconstruction is imperfect.
5. What must be recomputed in the OUT world if covariance and mask depend on subjects? Why is LOO sensitivity on a single dataset insufficient for a global DP claim?